In [ ]:
import sys
import time
import requests
import datetime
import json
import traceback
import sqlite3
import os
import re
from PyQt5.QtCore import QThread, pyqtSignal, Qt, QTimer, QDateTime
from PyQt5.QtWidgets import *
from PyQt5.QtGui import QColor, QFont, QIcon, QPalette, QLinearGradient, QBrush
import pyqtgraph as pg
import numpy as np

# ============================================
# دیتابیس برای ذخیره تاریخچه
# ============================================
class DatabaseManager:
    def __init__(self, db_name="market_data.db"):
        self.db_name = db_name
        self.init_db()
    
    def init_db(self):
        conn = sqlite3.connect(self.db_name)
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS prices (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                symbol TEXT NOT NULL,
                price REAL NOT NULL,
                timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
            )
        ''')
        conn.commit()
        conn.close()
    
    def save_price(self, symbol, price):
        conn = sqlite3.connect(self.db_name)
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO prices (symbol, price) VALUES (?, ?)",
            (symbol, price)
        )
        conn.commit()
        conn.close()
    
    def get_history(self, symbol, limit=100):
        conn = sqlite3.connect(self.db_name)
        cursor = conn.cursor()
        cursor.execute(
            "SELECT price, timestamp FROM prices WHERE symbol = ? ORDER BY timestamp DESC LIMIT ?",
            (symbol, limit)
        )
        data = cursor.fetchall()
        conn.close()
        return list(reversed(data))

# ============================================
# استخراج دیتا از چندین منبع معتبر
# ============================================
class NinjaScraperPro:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Accept-Language': 'en-US,en;q=0.9,fa;q=0.8',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
        })
        
        # منبع‌های معتبر برای دریافت داده
        self.sources = [
            {
                'name': 'Gold API (Global)',
                'url': 'https://api.gold-api.com/price/XAU',
                'type': 'gold_api'
            },
            {
                'name': 'ExchangeRate-API',
                'url': 'https://api.exchangerate-api.com/v4/latest/USD',
                'type': 'exchange_api'
            },
            {
                'name': 'TGJU Alternative',
                'url': 'https://www.tgju.org/profile/price_dollar_rl',
                'type': 'html'
            }
        ]
        
        # داده‌های واقعی آخرین بار
        self.last_real_data = None
        
        # نمادهای پشتیبانی شده با قیمت‌های تقریبی واقعی
        self.symbols = {
            'price_dollar_rl': {'name': 'Dollar (USD)', 'color': '#00d4ff', 'unit': 'IRT'},
            'geram18': {'name': 'Gold 18k', 'color': '#ffd700', 'unit': 'IRT'},
            'geram24': {'name': 'Gold 24k', 'color': '#ffb347', 'unit': 'IRT'},
            'price_ons': {'name': 'Global Gold', 'color': '#ff6b35', 'unit': 'USD'},
            'price_silver_ons': {'name': 'Silver', 'color': '#c0c0c0', 'unit': 'USD'},
            'price_eur': {'name': 'Euro (EUR)', 'color': '#4fc3f7', 'unit': 'IRT'},
            'price_gbp': {'name': 'Pound (GBP)', 'color': '#81c784', 'unit': 'IRT'},
            'price_cad': {'name': 'CAD Dollar', 'color': '#ff8a65', 'unit': 'IRT'},
            'price_aud': {'name': 'AUD Dollar', 'color': '#ce93d8', 'unit': 'IRT'},
            'price_aed': {'name': 'AED Dirham', 'color': '#fff176', 'unit': 'IRT'},
        }
        
        # قیمت‌های واقعی برای داده‌های نمونه
        self.realistic_prices = {
            'Dollar (USD)': 928000,
            'Gold 18k': 4570000,
            'Gold 24k': 6093000,
            'Global Gold': 2895,
            'Silver': 32.8,
            'Euro (EUR)': 1015000,
            'Pound (GBP)': 1175000,
            'CAD Dollar': 685000,
            'AUD Dollar': 610000,
            'AED Dirham': 252000
        }
    
    def fetch(self, use_proxy=False):
        """دریافت داده از چندین منبع"""
        print("🔄 Fetching real market data...")
        current_time = datetime.datetime.now().strftime("%H:%M:%S")
        
        # امتحان کردن همه منابع
        for source in self.sources:
            try:
                print(f"📡 Trying: {source['name']}")
                
                proxies = None
                if use_proxy:
                    proxies = {
                        'http': 'socks5://127.0.0.1:1080',
                        'https': 'socks5://127.0.0.1:1080'
                    }
                
                response = self.session.get(source['url'], timeout=10, proxies=proxies)
                
                if response.status_code == 200:
                    if source['type'] in ['gold_api', 'exchange_api']:
                        data = response.json()
                        result = self.parse_api_data(data, source['type'])
                    else:
                        result = self.parse_html_data(response.text)
                    
                    if result and len(result) > 0:
                        print(f"✅ Data received from: {source['name']}")
                        self.last_real_data = result
                        return result
                        
            except Exception as e:
                print(f"⚠️ Failed from {source['name']}: {str(e)[:50]}")
                continue
        
        # اگر همه منابع شکست خوردند، از داده‌های کش استفاده کن
        if self.last_real_data:
            print("📦 Using cached data")
            return self.last_real_data
        
        # در نهایت داده‌های نمونه واقع‌گرایانه
        print("⚠️ Using realistic sample data")
        return self.get_realistic_sample()
    
    def parse_api_data(self, data, data_type):
        """پارس داده از APIهای مختلف"""
        results = []
        current_time = datetime.datetime.now().strftime("%H:%M:%S")
        
        try:
            if data_type == 'gold_api':
                # دریافت قیمت طلا
                price = data.get('price', 0)
                if price:
                    results.append({
                        'symbol': 'Global Gold',
                        'price': float(price),
                        'unit': 'USD',
                        'color': '#ff6b35',
                        'time': current_time,
                        'change': '0'
                    })
                    # برآورد قیمت نقره (نسبت تقریبی 1:80)
                    results.append({
                        'symbol': 'Silver',
                        'price': float(price) / 80,
                        'unit': 'USD',
                        'color': '#c0c0c0',
                        'time': current_time,
                        'change': '0'
                    })
            
            elif data_type == 'exchange_api':
                # دریافت نرخ ارزها
                rates = data.get('rates', {})
                for code, name in [('EUR', 'Euro (EUR)'), ('GBP', 'Pound (GBP)')]:
                    if code in rates:
                        results.append({
                            'symbol': name,
                            'price': rates[code],
                            'unit': 'USD',
                            'color': self.symbols.get(f'price_{code.lower()}', {}).get('color', '#ffffff'),
                            'time': current_time,
                            'change': '0'
                        })
        
        except Exception as e:
            print(f"API parse error: {e}")
        
        return results if results else None
    
    def parse_html_data(self, html_content):
        """پارس داده از HTML"""
        try:
            # الگوهای مختلف برای استخراج قیمت
            patterns = [
                r'data-price="([\d,]+)"',
                r'price-value="([\d,]+)"',
                r'<span[^>]*class="[^"]*price[^"]*"[^>]*>([\d,]+)</span>',
                r'(\d{1,3}(?:,\d{3})*\.?\d*)\s*(?:تومان|ریال|دلار)'
            ]
            
            all_prices = []
            for pattern in patterns:
                matches = re.findall(pattern, html_content)
                if matches:
                    for match in matches:
                        try:
                            price = float(match.replace(',', ''))
                            if price > 0 and price < 1000000000:  # محدوده منطقی
                                all_prices.append(price)
                        except:
                            pass
            
            # اگر قیمت‌هایی پیدا شد، آنها را برگردان
            if all_prices:
                results = []
                current_time = datetime.datetime.now().strftime("%H:%M:%S")
                symbols = list(self.realistic_prices.keys())[:len(all_prices)]
                
                for i, price in enumerate(all_prices[:4]):
                    if i < len(symbols):
                        results.append({
                            'symbol': symbols[i],
                            'price': price,
                            'unit': 'IRT',
                            'color': self.symbols.get(list(self.symbols.keys())[i], {}).get('color', '#ffffff'),
                            'time': current_time,
                            'change': '0'
                        })
                
                return results if results else None
            
            return None
            
        except Exception as e:
            print(f"HTML parse error: {e}")
            return None
    
    def get_realistic_sample(self):
        """داده‌های نمونه با قیمت‌های واقعی"""
        current_time = datetime.datetime.now().strftime("%H:%M:%S")
        return [
            {'symbol': 'Dollar (USD)', 'price': 928000, 'unit': 'IRT', 'color': '#00d4ff', 'time': current_time, 'change': '+0.5'},
            {'symbol': 'Gold 18k', 'price': 4570000, 'unit': 'IRT', 'color': '#ffd700', 'time': current_time, 'change': '+0.3'},
            {'symbol': 'Gold 24k', 'price': 6093000, 'unit': 'IRT', 'color': '#ffb347', 'time': current_time, 'change': '+0.4'},
            {'symbol': 'Global Gold', 'price': 2895, 'unit': 'USD', 'color': '#ff6b35', 'time': current_time, 'change': '-0.2'},
            {'symbol': 'Silver', 'price': 32.8, 'unit': 'USD', 'color': '#c0c0c0', 'time': current_time, 'change': '+0.8'},
            {'symbol': 'Euro (EUR)', 'price': 1015000, 'unit': 'IRT', 'color': '#4fc3f7', 'time': current_time, 'change': '+0.1'},
            {'symbol': 'Pound (GBP)', 'price': 1175000, 'unit': 'IRT', 'color': '#81c784', 'time': current_time, 'change': '+0.2'},
            {'symbol': 'CAD Dollar', 'price': 685000, 'unit': 'IRT', 'color': '#ff8a65', 'time': current_time, 'change': '+0.0'},
        ]

# ============================================
# Worker پیشرفته
# ============================================
class DataWorkerPro(QThread):
    data_signal = pyqtSignal(list)
    status_signal = pyqtSignal(dict)
    error_signal = pyqtSignal(str)
    progress_signal = pyqtSignal(int)
    
    def __init__(self, use_proxy=False):
        super().__init__()
        self.scraper = NinjaScraperPro()
        self.db = DatabaseManager()
        self.running = True
        self.use_proxy = use_proxy
        
    def run(self):
        while self.running:
            try:
                # دریافت داده
                data = self.scraper.fetch(use_proxy=self.use_proxy)
                
                if data and len(data) > 0:
                    # ذخیره در دیتابیس
                    for item in data:
                        self.db.save_price(item['symbol'], item['price'])
                    
                    # ارسال داده
                    self.data_signal.emit(data)
                    self.status_signal.emit({
                        'online': True,
                        'count': len(data),
                        'time': datetime.datetime.now().strftime("%H:%M:%S")
                    })
                else:
                    self.status_signal.emit({
                        'online': False,
                        'count': 0,
                        'time': datetime.datetime.now().strftime("%H:%M:%S")
                    })
                
                time.sleep(5)  # هر 5 ثانیه یکبار
                
            except Exception as e:
                print(f"Worker error: {e}")
                self.error_signal.emit(str(e))
                time.sleep(10)
    
    def stop(self):
        self.running = False

# ============================================
# ویجت کارت قیمت
# ============================================
class PriceCard(QFrame):
    def __init__(self, symbol, price, unit, color, change='0'):
        super().__init__()
        self.symbol = symbol
        self.price = price
        self.unit = unit
        self.color = color
        self.change = change
        self.init_ui()
    
    def init_ui(self):
        self.setFixedSize(160, 100)
        self.setStyleSheet(f"""
            QFrame {{
                background: qlineargradient(x1:0, y1:0, x2:0, y2:1,
                    stop:0 #1a2332, stop:1 #0f172a);
                border-radius: 10px;
                border: 1px solid {self.color};
            }}
        """)
        
        layout = QVBoxLayout(self)
        layout.setSpacing(2)
        
        # اسم نماد
        label_symbol = QLabel(self.symbol)
        label_symbol.setFont(QFont("Segoe UI", 9, QFont.Bold))
        label_symbol.setStyleSheet(f"color: {self.color};")
        layout.addWidget(label_symbol)
        
        # قیمت
        label_price = QLabel(f"{self.price:,.0f}")
        label_price.setFont(QFont("Consolas", 14, QFont.Bold))
        label_price.setStyleSheet("color: #ffffff;")
        layout.addWidget(label_price)
        
        # واحد و تغییرات
        info_layout = QHBoxLayout()
        label_unit = QLabel(self.unit)
        label_unit.setStyleSheet("color: #64748b; font-size: 9px;")
        info_layout.addWidget(label_unit)
        
        change_color = "#10b981" if not self.change.startswith('-') else "#ef4444"
        label_change = QLabel(f"{self.change}%")
        label_change.setStyleSheet(f"color: {change_color}; font-size: 9px; font-weight: bold;")
        info_layout.addWidget(label_change)
        info_layout.addStretch()
        
        layout.addLayout(info_layout)

# ============================================
# رابط کاربری اصلی - نسخه کامل
# ============================================
class NinjaTraderPro(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("NINJA ALGO-TRADER V6.0 - PROFESSIONAL EDITION")
        self.setGeometry(50, 50, 1400, 900)
        
        # متغیرها
        self.data_history = {}
        self.max_history = 200
        self.chart_lines = {}
        self.is_proxy_enabled = False
        
        # دیتابیس
        self.db = DatabaseManager()
        
        self.setup_ui()
        self.apply_theme()
        
        # شروع worker
        self.start_worker()
        
        # تایمر
        self.timer = QTimer()
        self.timer.timeout.connect(self.update_clock)
        self.timer.start(1000)
    
    def start_worker(self):
        self.worker = DataWorkerPro(use_proxy=self.is_proxy_enabled)
        self.worker.data_signal.connect(self.on_data_received)
        self.worker.status_signal.connect(self.on_status_changed)
        self.worker.error_signal.connect(self.show_error)
        self.worker.start()
    
    def setup_ui(self):
        central = QWidget()
        self.setCentralWidget(central)
        main_layout = QVBoxLayout(central)
        main_layout.setSpacing(10)
        main_layout.setContentsMargins(15, 15, 15, 15)
        
        # === Header ===
        header = self.create_header()
        main_layout.addWidget(header)
        
        # === Price Cards ===
        self.cards_frame = QFrame()
        self.cards_frame.setStyleSheet("""
            QFrame {
                background-color: transparent;
                border: none;
            }
        """)
        self.cards_layout = QHBoxLayout(self.cards_frame)
        self.cards_layout.setSpacing(10)
        main_layout.addWidget(self.cards_frame)
        
        # === Main Content ===
        content = QHBoxLayout()
        
        # Left: Table
        left_panel = self.create_table_panel()
        content.addWidget(left_panel, 1)
        
        # Right: Chart
        right_panel = self.create_chart_panel()
        content.addWidget(right_panel, 2)
        
        main_layout.addLayout(content, 1)
        
        # === Status Bar ===
        self.status_bar = self.statusBar()
        self.status_bar.setStyleSheet("""
            QStatusBar {
                background-color: #0f172a;
                color: #94a3b8;
                border-top: 1px solid #1e293b;
                padding: 5px;
            }
        """)
    
    def create_header(self):
        header = QFrame()
        header.setFixedHeight(70)
        header.setStyleSheet("""
            QFrame {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:0,
                    stop:0 #0f172a, stop:1 #1a2332);
                border-radius: 12px;
                border: 1px solid #1e293b;
            }
        """)
        layout = QHBoxLayout(header)
        layout.setContentsMargins(20, 10, 20, 10)
        
        # Logo
        logo = QLabel("⚡ NINJA TRADER")
        logo.setFont(QFont("Segoe UI", 18, QFont.Bold))
        logo.setStyleSheet("color: #38bdf8;")
        layout.addWidget(logo)
        
        # Version
        version = QLabel("v6.0 PRO")
        version.setFont(QFont("Segoe UI", 10))
        version.setStyleSheet("color: #64748b;")
        layout.addWidget(version)
        
        layout.addStretch()
        
        # Status
        self.status_indicator = QLabel("●")
        self.status_indicator.setStyleSheet("color: #f59e0b; font-size: 24px;")
        layout.addWidget(self.status_indicator)
        
        self.status_text = QLabel("INITIALIZING...")
        self.status_text.setFont(QFont("Segoe UI", 10, QFont.Bold))
        self.status_text.setStyleSheet("color: #94a3b8;")
        layout.addWidget(self.status_text)
        
        # Proxy toggle
        self.proxy_btn = QPushButton("🌐 Proxy: OFF")
        self.proxy_btn.setCheckable(True)
        self.proxy_btn.clicked.connect(self.toggle_proxy)
        self.proxy_btn.setStyleSheet("""
            QPushButton {
                background-color: #1e293b;
                color: #f87171;
                border: 1px solid #f87171;
                border-radius: 6px;
                padding: 8px 15px;
                font-weight: bold;
            }
            QPushButton:checked {
                background-color: #1e293b;
                color: #10b981;
                border: 1px solid #10b981;
            }
        """)
        layout.addWidget(self.proxy_btn)
        
        # Refresh
        refresh_btn = QPushButton("🔄 Refresh")
        refresh_btn.clicked.connect(self.manual_refresh)
        refresh_btn.setStyleSheet("""
            QPushButton {
                background-color: #1e293b;
                color: #38bdf8;
                border: 1px solid #38bdf8;
                border-radius: 6px;
                padding: 8px 20px;
                font-weight: bold;
            }
            QPushButton:hover {
                background-color: #38bdf8;
                color: #0f172a;
            }
        """)
        layout.addWidget(refresh_btn)
        
        # Clock
        self.clock_label = QLabel()
        self.clock_label.setFont(QFont("Consolas", 12, QFont.Bold))
        self.clock_label.setStyleSheet("color: #60a5fa;")
        layout.addWidget(self.clock_label)
        
        return header
    
    def create_table_panel(self):
        panel = QFrame()
        panel.setFixedWidth(500)
        panel.setStyleSheet("""
            QFrame {
                background-color: #0f172a;
                border-radius: 12px;
                border: 1px solid #1e293b;
            }
        """)
        layout = QVBoxLayout(panel)
        layout.setContentsMargins(15, 15, 15, 15)
        
        # Title
        title = QLabel("📊 LIVE PRICES")
        title.setFont(QFont("Segoe UI", 13, QFont.Bold))
        title.setStyleSheet("color: #e2e8f0;")
        layout.addWidget(title)
        
        # Table
        self.table = QTableWidget(0, 5)
        self.table.setHorizontalHeaderLabels(["#", "Symbol", "Price", "Unit", "Change"])
        self.table.setStyleSheet("""
            QTableWidget {
                background-color: #0f172a;
                color: #e2e8f0;
                gridline-color: #1e293b;
                border: none;
                font-size: 12px;
            }
            QTableWidget::item:selected {
                background-color: #1e293b;
            }
            QHeaderView::section {
                background-color: #1a2332;
                color: #94a3b8;
                padding: 8px;
                font-weight: bold;
                border: none;
            }
        """)
        self.table.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        self.table.setAlternatingRowColors(True)
        layout.addWidget(self.table)
        
        # Stats
        self.stats_label = QLabel("📈 Last Update: -- | Symbols: 0")
        self.stats_label.setStyleSheet("color: #64748b; font-size: 11px;")
        layout.addWidget(self.stats_label)
        
        return panel
    
    def create_chart_panel(self):
        panel = QFrame()
        panel.setStyleSheet("""
            QFrame {
                background-color: #0f172a;
                border-radius: 12px;
                border: 1px solid #1e293b;
            }
        """)
        layout = QVBoxLayout(panel)
        layout.setContentsMargins(15, 15, 15, 15)
        
        # Title with controls
        header = QHBoxLayout()
        title = QLabel("📈 MARKET CHART")
        title.setFont(QFont("Segoe UI", 13, QFont.Bold))
        title.setStyleSheet("color: #e2e8f0;")
        header.addWidget(title)
        header.addStretch()
        
        # Symbol selector
        self.symbol_selector = QComboBox()
        self.symbol_selector.setStyleSheet("""
            QComboBox {
                background-color: #1a2332;
                color: #e2e8f0;
                border: 1px solid #2d3b4f;
                border-radius: 6px;
                padding: 5px;
                min-width: 120px;
            }
            QComboBox::drop-down {
                border: none;
            }
        """)
        self.symbol_selector.addItems(['All', 'Dollar (USD)', 'Gold 18k', 'Gold 24k', 'Global Gold', 'Silver', 'Euro (EUR)'])
        self.symbol_selector.currentTextChanged.connect(self.update_chart)
        header.addWidget(self.symbol_selector)
        
        # Time range
        self.time_range = QComboBox()
        self.time_range.setStyleSheet("""
            QComboBox {
                background-color: #1a2332;
                color: #e2e8f0;
                border: 1px solid #2d3b4f;
                border-radius: 6px;
                padding: 5px;
                min-width: 80px;
            }
        """)
        self.time_range.addItems(['1min', '5min', '15min', '30min', '1hour'])
        self.time_range.currentTextChanged.connect(self.update_chart)
        header.addWidget(self.time_range)
        
        layout.addLayout(header)
        
        # Chart
        self.plot = pg.PlotWidget()
        self.plot.setBackground('#0f172a')
        self.plot.showGrid(x=True, y=True, alpha=0.15)
        self.plot.setLabel('left', 'Price', units='')
        self.plot.setLabel('bottom', 'Time')
        
        # Axis styling
        self.plot.getAxis('left').setPen(pg.mkPen(color='#475569', width=1))
        self.plot.getAxis('bottom').setPen(pg.mkPen(color='#475569', width=1))
        self.plot.getAxis('left').setTextPen(pg.mkPen(color='#94a3b8'))
        self.plot.getAxis('bottom').setTextPen(pg.mkPen(color='#94a3b8'))
        
        layout.addWidget(self.plot)
        
        # Legend
        self.legend_frame = QFrame()
        self.legend_frame.setFixedHeight(35)
        self.legend_frame.setStyleSheet("""
            QFrame {
                background-color: #1a2332;
                border-radius: 8px;
                border: 1px solid #2d3b4f;
            }
        """)
        self.legend_layout = QHBoxLayout(self.legend_frame)
        self.legend_layout.setContentsMargins(10, 5, 10, 5)
        self.legend_items = {}
        layout.addWidget(self.legend_frame)
        
        return panel
    
    def apply_theme(self):
        self.setStyleSheet("""
            QMainWindow {
                background-color: #020617;
            }
            QScrollBar:vertical {
                background-color: #0f172a;
                width: 10px;
                border-radius: 5px;
            }
            QScrollBar::handle:vertical {
                background-color: #1e293b;
                border-radius: 5px;
            }
            QScrollBar::add-line:vertical, QScrollBar::sub-line:vertical {
                height: 0px;
            }
        """)
    
    # ============================================
    # توابع اصلی
    # ============================================
    
    def on_data_received(self, data):
        """دریافت داده جدید"""
        print(f"📊 Received {len(data)} items")
        
        # ذخیره تاریخچه
        for item in data:
            symbol = item['symbol']
            if symbol not in self.data_history:
                self.data_history[symbol] = []
            
            self.data_history[symbol].append({
                'price': item['price'],
                'time': datetime.datetime.now()
            })
            
            if len(self.data_history[symbol]) > self.max_history:
                self.data_history[symbol] = self.data_history[symbol][-self.max_history:]
        
        # بروزرسانی کارت‌ها
        self.update_cards(data)
        
        # بروزرسانی جدول
        self.update_table(data)
        
        # بروزرسانی نمودار
        self.update_chart()
        
        # بروزرسانی آمار
        self.stats_label.setText(f"📈 Last Update: {data[0]['time'] if data else '--'} | Symbols: {len(data)}")
    
    def update_cards(self, data):
        """بروزرسانی کارت‌های قیمت"""
        # پاک کردن کارت‌های قبلی
        for i in reversed(range(self.cards_layout.count())):
            widget = self.cards_layout.itemAt(i).widget()
            if widget:
                widget.deleteLater()
        
        # اضافه کردن کارت‌های جدید
        for item in data[:8]:  # حداکثر 8 کارت
            card = PriceCard(
                item['symbol'],
                item['price'],
                item.get('unit', 'IRT'),
                item.get('color', '#ffffff'),
                item.get('change', '0')
            )
            self.cards_layout.addWidget(card)
    
    def update_table(self, data):
        """بروزرسانی جدول"""
        self.table.setRowCount(0)
        
        # مرتب‌سازی بر اساس قیمت
        sorted_data = sorted(data, key=lambda x: x['price'], reverse=True)
        
        for idx, item in enumerate(sorted_data, 1):
            row = self.table.rowCount()
            self.table.insertRow(row)
            
            # شماره
            num = QTableWidgetItem(str(idx))
            num.setForeground(QColor("#64748b"))
            num.setTextAlignment(Qt.AlignCenter)
            
            # Symbol
            symbol = QTableWidgetItem(item['symbol'])
            symbol.setForeground(QColor(item.get('color', '#ffffff')))
            symbol.setFont(QFont("Segoe UI", 10, QFont.Bold))
            
            # Price
            price = QTableWidgetItem(f"{item['price']:,.0f}")
            price.setForeground(QColor("#10b981"))
            price.setFont(QFont("Consolas", 11, QFont.Bold))
            price.setTextAlignment(Qt.AlignRight | Qt.AlignVCenter)
            
            # Unit
            unit = QTableWidgetItem(item.get('unit', 'IRT'))
            unit.setForeground(QColor("#64748b"))
            
            # Change
            change = item.get('change', '0')
            change_color = "#10b981" if not change.startswith('-') else "#ef4444"
            change_item = QTableWidgetItem(f"{change}%")
            change_item.setForeground(QColor(change_color))
            change_item.setFont(QFont("Segoe UI", 9, QFont.Bold))
            
            self.table.setItem(row, 0, num)
            self.table.setItem(row, 1, symbol)
            self.table.setItem(row, 2, price)
            self.table.setItem(row, 3, unit)
            self.table.setItem(row, 4, change_item)
    
    def update_chart(self):
        """بروزرسانی نمودار"""
        self.plot.clear()
        self.clear_legend()
        
        selected = self.symbol_selector.currentText()
        
        for symbol, history in self.data_history.items():
            if selected != 'All' and symbol != selected:
                continue
            
            if len(history) < 2:
                continue
            
            # استخراج داده
            prices = [h['price'] for h in history]
            times = list(range(len(prices)))
            
            # رنگ
            color = '#38bdf8'
            # پیدا کردن رنگ از داده‌های قبلی
            for item in NinjaScraperPro().symbols.values():
                if item['name'] == symbol:
                    color = item['color']
                    break
            
            # رسم خط
            pen = pg.mkPen(color=color, width=2)
            line = self.plot.plot(times, prices, pen=pen)
            
            # نقاط
            scatter = pg.ScatterPlotItem(
                size=4,
                brush=pg.mkBrush(color),
                pen=pg.mkPen(color=color, width=0.5)
            )
            scatter.setData(times, prices)
            self.plot.addItem(scatter)
            
            # اضافه به Legend
            self.add_legend_item(symbol, color)
    
    def add_legend_item(self, symbol, color):
        """اضافه کردن آیتم به Legend"""
        label = QLabel(f"● {symbol}")
        label.setStyleSheet(f"color: {color}; font-size: 10px; font-weight: bold;")
        self.legend_layout.addWidget(label)
        self.legend_items[symbol] = label
    
    def clear_legend(self):
        """پاک کردن Legend"""
        for item in self.legend_items.values():
            item.deleteLater()
        self.legend_items.clear()
    
    def on_status_changed(self, status):
        """تغییر وضعیت"""
        if status['online']:
            self.status_indicator.setStyleSheet("color: #10b981; font-size: 24px;")
            self.status_text.setText(f"ONLINE • {status['count']} symbols")
            self.status_text.setStyleSheet("color: #10b981;")
        else:
            self.status_indicator.setStyleSheet("color: #f59e0b; font-size: 24px;")
            self.status_text.setText("CONNECTING...")
            self.status_text.setStyleSheet("color: #f59e0b;")
    
    def update_clock(self):
        """بروزرسانی ساعت"""
        current = QDateTime.currentDateTime()
        self.clock_label.setText(f"🕐 {current.toString('HH:mm:ss')}")
    
    def toggle_proxy(self):
        """تغییر وضعیت پروکسی"""
        self.is_proxy_enabled = self.proxy_btn.isChecked()
        self.proxy_btn.setText(f"🌐 Proxy: {'ON' if self.is_proxy_enabled else 'OFF'}")
        
        # راه‌اندازی مجدد worker
        self.worker.stop()
        self.worker.wait()
        self.start_worker()
    
    def manual_refresh(self):
        """رفرش دستی"""
        print("🔄 Manual refresh...")
        self.status_text.setText("REFRESHING...")
        self.status_text.setStyleSheet("color: #60a5fa;")
        
        self.worker.stop()
        self.worker.wait()
        self.start_worker()
    
    def show_error(self, message):
        """نمایش خطا"""
        QMessageBox.warning(self, "Error", f"Data fetch error:\n{message}")
    
    def closeEvent(self, event):
        """بستن برنامه"""
        print("🛑 Shutting down...")
        self.worker.stop()
        self.worker.wait()
        event.accept()

# ============================================
# اجرا
# ============================================
if __name__ == "__main__":
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    
    # تنظیم فونت
    font = QFont("Segoe UI", 9)
    app.setFont(font)
    
    # ایجاد پنجره
    window = NinjaTraderPro()
    window.show()
    
    print("=" * 50)
    print("⚡ NINJA ALGO-TRADER V6.0 PRO")
    print("📡 Waiting for real market data...")
    print("=" * 50)
    
    sys.exit(app.exec_())

🔄 Fetching real market data...
📡 Trying: Gold API (Global)
⚡ NINJA ALGO-TRADER V6.0 PRO
📡 Waiting for real market data...
✅ Data received from: Gold API (Global)
📊 Received 2 items
🔄 Fetching real market data...
📡 Trying: Gold API (Global)
✅ Data received from: Gold API (Global)
📊 Received 2 items
🔄 Fetching real market data...
📡 Trying: Gold API (Global)
✅ Data received from: Gold API (Global)
📊 Received 2 items
🔄 Manual refresh...
🔄 Fetching real market data...
📡 Trying: Gold API (Global)
🔄 Manual refresh...
✅ Data received from: Gold API (Global)
🔄 Fetching real market data...
📡 Trying: Gold API (Global)
📊 Received 2 items
✅ Data received from: Gold API (Global)
📊 Received 2 items
🔄 Fetching real market data...
📡 Trying: Gold API (Global)
✅ Data received from: Gold API (Global)
📊 Received 2 items
🔄 Fetching real market data...
📡 Trying: Gold API (Global)
✅ Data received from: Gold API (Global)
📊 Received 2 items
🔄 Fetching real market data...
📡 Trying: Gold API (Global)
✅ Data recei